In [132]:
from filterpy.kalman import KalmanFilter
from project_brain_decoder.train_gru import data_folder
from src.project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
import numpy as np
import gc

In [133]:
files = list(data_folder.glob("*.nwb"))
train_files = files[:10]
neural_list = []
targets_list = []
X_list = []
y_list = []
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [134]:
for file in train_files:
    loaded_file = load_nwb(file_path=file)
    spiking = loaded_file["neural_spiking_band"]
    threshold = loaded_file["neural_threshold_crossings"]
    neural = np.concatenate([spiking, threshold], axis=1)
    index = loaded_file["target_index_velocity"]
    mrs = loaded_file["target_mrs_velocity"]
    targets = np.column_stack([index, mrs])
    neural_list.append(neural)
    targets_list.append(targets)
neural_all = np.concatenate(neural_list, axis=0)
targets_all = np.concatenate(targets_list, axis=0)
neural_scaler.fit(neural_all)
targets_scaler.fit(targets_all)
del neural_all, targets_all
gc.collect()

4038

In [135]:
for neural, targets in zip(neural_list, targets_list):
    scaled_n = neural_scaler.transform(neural)
    scaled_t = targets_scaler.transform(targets)
    X_list.append(scaled_n)
    y_list.append(scaled_t)
X_train = np.concatenate(X_list)
y_train = np.concatenate(y_list)

In [162]:
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train)

In [163]:
# Predicting velocity at t using velocity at t+1 (2x2)
vel_prev = y_train[:-1] # x_{t-1}
vel_curr = y_train[1:] # x_t
F = (vel_curr.T @ vel_prev) @ np.linalg.inv(vel_prev.T @ vel_prev)

In [164]:
# Mapping velocity to neural activity (192x2)
H = np.linalg.lstsq(y_train, X_train_pca, rcond=None)[0].T

In [165]:
H.shape

(75, 2)

In [166]:
# Processing noise - residuals of state transition (2x2)
state_res = vel_curr - (F @ vel_prev.T).T
Q = np.cov(state_res.T)

In [167]:
# Observation noise - residuals of observation model (192x192), diagonal
obs_res = X_train_pca - (H @ y_train.T).T
R = np.diag(np.var(obs_res, axis=0))

In [174]:
f = KalmanFilter(dim_x=2, dim_z=50)
f.x = np.array([[0.], [0.]])
f.P = 1000 * np.eye(2)
f.F = F
f.H = H
f.Q = Q
f.R = R

In [175]:
val_file = files[10]
loaded_val = load_nwb(val_file)
spiking_v = loaded_val["neural_spiking_band"]
thresh_v = loaded_val["neural_threshold_crossings"]
neural_val = neural_scaler.transform(np.concatenate([spiking_v, thresh_v], axis=1))
neural_val_pca = pca.transform(neural_val)

In [176]:
index_vel = loaded_val["target_index_velocity"]
mrs_vel = loaded_val["target_mrs_velocity"]
y_val = np.column_stack([index_vel, mrs_vel])

In [177]:
predictions = []
z = neural_val_pca
T = len(neural_val_pca)

In [178]:
for t in range(T):
    f.predict()
    f.update(z[t].reshape(-1, 1)) # filterpy
    predictions.append(f.x.copy())

In [179]:
preds = np.array(predictions).squeeze() # (T, 2)
preds_unscaled = targets_scaler.inverse_transform(preds)

In [180]:
y_pred_from_H = (np.linalg.pinv(H) @ neural_val_pca.T).T
r2_H = r2_score(targets_scaler.transform(y_val), y_pred_from_H, multioutput="raw_values")
print(f"H-only R² index: {r2_H[0]:.4f}, R² mrs: {r2_H[1]:.4f}")

H-only R² index: -14.9370, R² mrs: -6.6435


In [181]:
r2 = r2_score(y_true=y_val, y_pred=preds_unscaled, multioutput="raw_values")
print(f"R² index: {r2[0]:.4f}, R² mrs: {r2[1]:.4f}")

R² index: 0.0956, R² mrs: 0.0215


In [182]:
# # Least squares: neural -> velocity (for diagnostic)
# W = np.linalg.lstsq(X_train, y_train, rcond=None)[0] # (192, 2)
# y_pred_W = neural_val @ W
# r2_W = r2_score(targets_scaler.transform(y_val), y_pred_W, multioutput="raw_values")
# print(f"Direct decoder R² index: {r2_W[0]:.4f}, R² mrs: {r2_W[1]:.4f}")